## Medications
Notes: [WHO ATC Codes](https://atcddd.fhi.no/atc_ddd_index/)

Some background information:
# ATC code
The ATC is a structured, hierarchical code. The code start generically and becomes increasingly more detailed.
The ATC structure is described as follows, take an example ```A02BC01```
- *A*: ATC 1st level -> **Main group**, in this case *Alimentary tract and metabolism*
- *02*: ATC 2nd level -> **Therapeutic group**, in this case *Drugs for acid-related disorders*
- *B*: ATC 3th level -> **Pharmacological group**, in this case *Drugs for peptic ulcer and gastro-oesophageal reflux disease*
- *C*: ATC 4th level -> **Chemical group**, in this case *Proton pump inhibitors*
- *01*: ATC 5th level -> **Active substance**, in this case *Omeprazole*

Main groups:
```
Alimentary tract and metabolism (A)
Blood and blood forming organs (B)
Cardiovascular system (C)
Dermatologicals (D)
Genito urinary system and sex hormones (G)
Systemic hormonal preparations, excl. sex hormones and insulins (H)
Antiinfectives for systemic use (J)
Antineoplastic and immunomodulating agents (L)
Musculo-skeletal system (M)
Nervous system (N)
Antiparasitic products, insecticides and repellents (P)
Respiratory system (R)
Sensory organs (S)
Various (V)
```

You can find ATC/CUI/SNOMED mappings here: ```T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS```

# The dataset contains more information than just the medication

* ```Med_Mednaam```: full name of medication including dosage
* ```category_display_original```: 
  * ```Opname```: during admission to hospital -> more acute situation
  * ```Thuis/verpleeghuis```: during stay in (care) home -> less acute situation
* ```dosInstr_doseQuantity_unit```: unit of dosage
* ```dosInstr_doseQuantity_value```: dosage value (measured in the unit of dosage)
* ```dosInstr_frequency_value```: frequency (per day)
* ```dosInstr_frequency_unit```: frequency unit (day/month/year)
* ```dosInstr_PerBetwDos_value```: number of time units between between dosage intake-sessions
* ```dosInstr_PerBetwDosUnit_Orig```: time unit definition
* ```intendedRoute_display```: method in of ingestion
* ```status_display_original```: first recipe/continuation/stop
* ```usageDuration_valueDuration_valu```: prescribed duration
  
To interpret the timing features, suppose;
* ```dosInstr_frequency_value = 4```
* ```dosInstr_frequency_unit = 'day'```
* ```dosInstr_PerBetwDos_value = 14```
* ```dosInstr_PerBetwDosUnit_Orig = 'day'```
Means: once per $14$ days the patient takes the medication $4$ times in a day.

If ```dosInstr_PerBetwDos_value``` is ```None``` then the medication is simply taken ```dosInstr_frequency_value``` per ```dosInstr_frequency_unit```.

# The problem: high cardinality

In our case we have about $900$ unique ATC codes. This is too high for any practical purpose, also there 
will be a lot of sparsity, most of the ATC values will be $0$ (assuming $1$ for presence, and $0$ for absence).

## Solutions and suggestions

### Make use of the hierarchical nature
The cardinality will increase if we add more ATC levels. Perhaps ATC level1+level2+level3 is enough and would allow basic one-hot encoding.
In fact, we can make feature combinations: ```level1+level2+level3, level1+level2, level1+level3, level2+level3```.

### Filter out the non-informative features

### One-hot encoding followed by dimension reduction

### Express each ATC code with a pre-trained embedding
* An aggregation of word embeddings: ```T:\laupodteam\AIOS\SUPPORTING_DATA\WordVectors```
* Document embeddings using SBERT: ```T:\laupodteam\AIOS\SUPPORTING_DATA\SentenceEmbedders```
* Graph embeddings: ```T:\laupodteam\AIOS\SUPPORTING_DATA\OntologyEmbeddings```
* A combination of the above

To make use of document/sentence embeddings as completely as possible maybe combine
```Med_Mednaam```, ```category_display_original```, ```intendedRoute_display```, ```status_display_original``` into a short narrative.

Also, perhaps make use of the 


### Use supervised feature encoding
* [NCA](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html)/[LDA](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html)
* [WoE](https://feature-engine.trainindata.com/en/latest/user_guide/encoding/WoEEncoder.html)

In [7]:
import pandas as pd
import os
import gc

input_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805'
output_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data'

In [8]:
ATC_CUI = pd.read_parquet(r'T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS\ATC_CUI.parquet')
CUI_DEF = pd.read_parquet(r'T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS\CUI_DEF.parquet')
ATC_STR = CUI_DEF.loc[CUI_DEF.LAT=='ENG'].merge(ATC_CUI, on='CUI', how='inner')
ATC_STR = ATC_STR.assign(STR=ATC_STR.STR.str.lower())
ATC_STR = ATC_STR[~ATC_STR.duplicated(subset=['CUI', 'ATC_code'])]

In [ ]:
ATC_STR.STR.unique().shape

In [4]:
CUI_CUI = pd.read_parquet(r'T:\laupodteam\AIOS\SUPPORTING_DATA\UMLS\CUI_CUI.parquet')

In [1]:
import pandas as pd
import os

output_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data'
os.chdir(output_data)

In [ ]:
# Caricamento del dataset
df_medicatie = pd.read_parquet(r'parquet\df_medicatie.parquet')

# Eventuale rimozione dei duplicati
df_medicatie.drop_duplicates(keep='first', inplace=True)

# Analisi per capire quali colonne hanno una percentuale di NaN
# inferiore ad una certa soglia
missing_percentage = df_medicatie.isnull().mean()
keep_cols = df_medicatie.loc[:, missing_percentage < 0.2].columns

print(f'{keep_cols}, # = {len(keep_cols)}\n')
print(f'Total number of records: {df_medicatie.shape[0]}\n')
print(f'Number of unique patients: {df_medicatie["studyId_0831"].nunique()}\n')
print(f'Number of missing ATC codes: {len(df_medicatie[df_medicatie["code5_ATC_code"].isna()].code5_ATC_code)}')

In [30]:
# Viene caricato il dizionario per gli ATC codes
dictionary = pd.read_csv('T:\laupodteam\AIOS\SUPPORTING_DATA\ATC\WHO ATC-DDD 2024-09-25.csv',sep='\t')

# Trova tutte le righe con NaN nella colonna specificata
missing_ATC = df_medicatie[df_medicatie['code5_ATC_code'].isna() | \
    df_medicatie['code5_ATC_display'].isna()].code2_HPK_display

# Rimozione di tutti i record il cui ATC code è mancante
df_medicatie = df_medicatie.dropna(subset=['code5_ATC_code', 'code5_ATC_display'])

# Inizializzazione della lista contenente le stringhe degli ATC codes
ATC_strings = [] 

for code in df_medicatie['code5_ATC_code']:
    # Inizializzare la stringa con 'Main group'
    string = 'Main group: ' 
    
    # Estrarre i valori dalle query e concatenarli, gestendo i risultati vuoti
    main_group = dictionary.query(f'atc_code == "{code[0]}"')
    if not main_group.empty:
        string += main_group.atc_name.values[0] + '. '
    
    therapeutic_group = dictionary.query(f'atc_code == "{code[0:3]}"')
    if not therapeutic_group.empty:
        string += 'Therapeutic group: ' + therapeutic_group.atc_name.values[0] + '. '
    
    pharmacological_group = dictionary.query(f'atc_code == "{code[0:4]}"')
    if not pharmacological_group.empty:
        string += 'Pharmacological group: ' + pharmacological_group.atc_name.values[0] + '. '
    
    chemical_group = dictionary.query(f'atc_code == "{code[0:5]}"')
    if not chemical_group.empty:
        string += 'Chemical group: ' + chemical_group.atc_name.values[0] + '. '
    
    active_substance = dictionary.query(f'atc_code == "{code}"')
    if not active_substance.empty:
        string += 'Active substance: ' + active_substance.atc_name.values[0]

    # Aggiungere la stringa alla lista
    ATC_strings.append(string)


""" for code in df_medicatie['code5_ATC_code']:
    string = str('Main group: ') + dictionary.query(f'atc_code == "{code[0]}"').atc_name.values[0] + str('. ') + \
        str('Therapeutic group: ') + dictionary.query(f'atc_code == "{code[0:3]}"').atc_name.values[0] + str('. ') + \
        str('Pharmacologial group: ') + dictionary.query(f'atc_code == "{code[0:4]}"').atc_name.values[0] + str('. ') + \
        str('Chemical group: ') + dictionary.query(f'atc_code == "{code[0:5]}"').atc_name.values[0] + str('. ') + \
        str('Active substance: ') + dictionary.query(f'atc_code == "{code}"').atc_name.values[0]
    
    ATC_strings.append(string) """

# La lista contenente le stringhe degli ATC codes viene inserita all'interno del dataframe
df_medicatie.insert(df_medicatie.columns.get_loc('code5_ATC_code') + 1, 'ATC_strings', ATC_strings)

# Salvo il dataframe nel percorso 'output_data'
df_medicatie.to_parquet('df_medicatie_ATC.parquet')

## Start from here (30/09/2024)

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer

df_medicatie = pd.read_parquet(r'L:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\df_medicatie_ATC.parquet')
df_medicatie = df_medicatie.reset_index(drop=True)

# Creazione della lista contenente le stringhe relative agli ATC codes
ATC_strings = df_medicatie["ATC_strings"].to_list()
print(f'Unique ATC codes: {len(df_medicatie["ATC_strings"].unique())}')

# Viene istanziato il modello di embedding
model = SentenceTransformer("FremyCompany/BioLORD-2023")

In [ ]:
# Embedding degli ATC codes (caldamente consigliato avere una GPU)
embeddings = model.encode(ATC_strings,
                        batch_size=768,
                        show_progress_bar=True)

In [2]:
embeddings = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\embeddings.parquet')

RS = 42

In [3]:
colori_exa = [
    "#FF5733",  # Rosso Aranciato
    "#33FF57",  # Verde Lime
    "#3357FF",  # Blu
    "#FF33A6",  # Rosa
    "#FF33D4",  # Magenta
    "#33FFF9",  # Ciano
    "#FFBF00",  # Giallo Oro
    "#FF7F50",  # Corallo
    "#8A2BE2",  # Blu Violetta
    "#FFD700",  # Giallo
    "#7FFF00",  # Verde Chartreuse
    "#D2691E",  # Cioccolato
    "#FFF500",  # Arancione Rosso
    "#6495ED"   # Blu Azzurro
]

letters = df_medicatie['code5_ATC_code'].sort_values().str[0].unique()

# Crea una mappatura gruppo-colore in formato RGB
color_map = {gruppo: colore for gruppo, colore in zip(letters, colori_exa)}

In [4]:
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

### 2 Components PCA

In [11]:
from sklearn.decomposition import PCA


pca = PCA(n_components=2, random_state=RS)

embeddings_pca = pca.fit_transform(embeddings)

Plots

In [ ]:
plt.figure(figsize=[12, 12])

# plotta le prime due colonne del DataFrame
plt.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1], \
    c=[color_map[lettera] for lettera in df_medicatie['code5_ATC_code'].str[0]])

# nomina i due assi
plt.xlabel('PC1', fontsize=18)
plt.ylabel('PC2', fontsize=18)

# crea la legenda
legend_elements = []
for lettera, colore in sorted(color_map.items()):
    legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', label=lettera, markerfacecolor=colore, markersize=10))

plt.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.show()

In [12]:
# Converti embeddings_pca in un DataFrame Pandas
df = pd.DataFrame(embeddings_pca, columns=['PC1', 'PC2'])

df['group'] = df_medicatie['code5_ATC_code'].str[0]

df = df.sort_values(by='group')

fig = px.scatter(df, x='PC1', y='PC2', color='group', color_discrete_map=color_map)

# Imposto la grandezza dei marker
fig.update_traces(marker=dict(size=7))

# Rimuovi la griglia dagli assi
fig.update_layout(
    
    title=dict(
        text="Scatter plot with the first two Principal Components",  # Il testo del titolo
        x=0.5,  # Centra il titolo
        xanchor='center',  # Imposta il punto di ancoraggio al centro
        font=dict(size=24)
    ),
    
    legend=dict(title=dict(text='ATC Group')),
    
    xaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    ),
    
    yaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    )
)

fig.show(renderer='browser')

fig.write_image(r"C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\twoPCA.png")

In [ ]:
# Crea il box plot per PC1
fig = px.box(df, x='group', y='PC1', title='Box Plot di PC1 per Gruppo')
fig.show()


fig = px.box(df, x='group', y='PC2', title='Box Plot di PC1 per Gruppo', color_discrete_sequence=['#EF553B'])
fig.show()

In [ ]:
from scipy import stats

# Esegui ANOVA su PC1 per tutti i gruppi
f_stat, p_value = stats.f_oneway(*(df[df['group'] == lettera]['PC1'] for lettera in letters))

# Esegui ANOVA su PC1 per tutti i gruppi
f_stat_pc2, p_value_pc2 = stats.f_oneway(*(df[df['group'] == lettera]['PC2'] for lettera in letters))

print(f"Statistiche F: {f_stat:.2}, p-value: {p_value}")
print(f"Statistiche F: {f_stat_pc2:.2}, p-value: {p_value_pc2}")

### 3 Components PCA

In [90]:
from sklearn.decomposition import PCA


pca = PCA(n_components=3, random_state=RS)
embeddings_pca = pca.fit_transform(embeddings)

Plots

In [ ]:
fig = plt.figure(figsize=[12, 12])
ax = fig.add_subplot(111, projection='3d')

ax.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1], embeddings_pca[:, 2], \
    c=[color_map[lettera] for lettera in df_medicatie['code5_ATC_code'].str[0]])

ax.set_xlabel('PC1', fontsize=18)
ax.set_ylabel('PC2', fontsize=18)
ax.set_zlabel('PC3', fontsize=18)

ax.set_xlim(min(embeddings_pca[:, 0])-1, max(embeddings_pca[:, 0]+1))
ax.set_ylim(min(embeddings_pca[:, 1])-1, max(embeddings_pca[:, 1]+1))
ax.set_zlim(min(embeddings_pca[:, 2])-1, max(embeddings_pca[:, 2]+1))

ax.view_init(elev=30, azim=60)

legend_elements = []
for lettera, colore in sorted(color_map.items()):
    legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', label=lettera, markerfacecolor=colore, markersize=10))

plt.legend(handles=legend_elements, loc='upper right', fontsize=10)
plt.show()

In [93]:
# Converti embeddings_pca in un DataFrame Pandas
df = pd.DataFrame(embeddings_pca, columns=['PC1', 'PC2', 'PC3'])

df['group'] = df_medicatie['code5_ATC_code'].str[0]

df = df.sort_values(by='group')

# Usa il DataFrame per creare un grafico 3D scatter
fig = px.scatter_3d(df, x='PC1', y='PC2', z='PC3', color='group', color_discrete_map=color_map)

fig.update_layout(
    
    title=dict(
        text="Scatter plot with the first three Principal Components",  # Il testo del titolo
        x=0.5,  # Centra il titolo
        xanchor='center',  # Imposta il punto di ancoraggio al centro
        font=dict(size=24)
    ),
    
    legend=dict(
        title=dict(text='ATC Group'),
        x=1,
        y=0.75,
        xanchor='right',
        yanchor='top'
    ),
    
    scene=dict(
        xaxis=dict(title_font=dict(size=20)),
        
        yaxis=dict(title_font=dict(size=20)),
        
        zaxis=dict(title_font=dict(size=20)),
        
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.2)),
        
        aspectmode='cube',  # Imposta il rapporto di aspetto
    )
)

# Imposta la dimensione dei marker
fig.update_traces(
    
    marker=dict(size=3),
    showlegend=False
)

fig.show(renderer='browser')
fig.write_image(r"C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\threePCA.png")

### General PCA

In [ ]:
from sklearn.decomposition import PCA


pca = PCA(random_state=RS)
embeddings_pca = pca.fit_transform(embeddings)

exp_variance_ratio = pca.explained_variance_ratio_
exp_variance = pca.explained_variance_

exp_var_perc = [np.sum(exp_variance[:i+1]) for i in range(len(exp_variance))]/np.sum(exp_variance)

perc = 0
n_components = 0

while perc <= 0.9:
    perc += exp_variance_ratio[n_components]
    n_components += 1
    

embeddings_pca = pd.DataFrame(embeddings_pca[:, :n_components], columns=[f"PC{i+1}" for i in range(n_components)])

embeddings_pca.insert(0, 'studyId_0831', df_medicatie['studyId_0831'])
embeddings_pca.insert(1, 'ATC_group', df_medicatie['code5_ATC_code'].str[0])

embeddings_pca.to_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\embeddings_perc.parquet")

last_point = round(exp_var_perc[n_components]*100, 1)

# Plot della figura
plt.figure(figsize = [20, 10])
plt.plot(np.arange(1, n_components+1), exp_var_perc[:n_components]*100, 'o-')

# crea una linea orizzontale in corrispondenza dell'ultimo punto
plt.axhline(y=perc*100, color='r', linestyle='--')

plt.xticks(np.concatenate((np.arange(1, n_components-2, 10), [n_components])))
plt.yticks(np.concatenate((np.arange(20, 100, 20), [round(perc*100, 1)])))
plt.tick_params(axis='both', labelsize=16)

plt.xlabel('Principal Components', fontsize=20)
plt.ylabel('Explained Variance [%]', fontsize=20)

plt.title(f'{n_components} components to capture {round(perc*100, 1):.1f}% of the informative content', fontsize=20)

plt.show()


## Clustering
Viene utilizzato il dataset originale in seguito alla feature extraction con la PCA

In [47]:
from sklearn.cluster import KMeans
from sklearn import metrics


embeddings_perc = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\embeddings_perc.parquet')

In [57]:
metriche_clustering = [
    "adjusted_mutual_info_score",
    "adjusted_rand_score",
    "calinski_harabasz_score",
    "completeness_score",
#    "davies_bouldin_score",
    "fowlkes_mallows_score",
#    "homogeneity_score",
#    "silhouette_score",
]

n_iter = 100

df_metrics = np.empty([n_iter, len(metriche_clustering)])

# Estraggo i veri cluster e li trasformo in numeri
true_labels = pd.factorize(embeddings_perc['ATC_group'])[0]

# Tolgo le prime due colonne dal dataframe originale
embeddings = embeddings_perc.iloc[:, 2:]

for i in range(n_iter):
    # Applicazione del K-Means con 14 cluster
    kmeans = KMeans(n_clusters=14)
    
    # Salvo i cluster predetti dal k-means
    pred_labels = kmeans.fit_predict(embeddings)
    
    df_metrics[i, :] = np.array([
        metrics.adjusted_mutual_info_score(true_labels, pred_labels),
        metrics.adjusted_rand_score(true_labels, pred_labels),
        metrics.calinski_harabasz_score(embeddings, pred_labels),
        metrics.completeness_score(true_labels, pred_labels),
        metrics.fowlkes_mallows_score(true_labels, pred_labels)
        ])  

## Metriche di Clustering



### 1. **Adjusted Mutual Information (AMI)**
- **Descrizione**: Misura la quantità di informazione condivisa tra due cluster, correggendo per il numero di cluster casuali.
- **Valori**: I valori variano da 0 a 1.
  - **1**: Corrispondenza perfetta.
  - **0**: Nessuna corrispondenza.

### 2. **Adjusted Rand Index (ARI)**
- **Descrizione**: Confronta l'accordo tra due partizionamenti di un dataset (ad esempio, cluster predetti e veri), correggendo per le assegnazioni casuali. 
- **Valori**: I valori variano da -1 a 1.
  - **1**: Corrispondenza perfetta tra i cluster.
  - **0**: Nessuna corrispondenza migliore di quella casuale.
  - **Valori negativi**: Corrispondenza peggiore di quella casuale.

### 3. **Calinski-Harabasz Score**
- **Descrizione**: Misura il rapporto tra la varianza inter-cluster e intra-cluster. Valuta la separazione e la compattezza dei cluster.
- **Valori**: Non ha un limite superiore, ma valori più alti indicano una migliore qualità dei cluster.

### 4. **Completeness Score**
- **Descrizione**: Valuta se tutti i punti di una stessa classe sono assegnati al medesimo cluster.
- **Valori**: I valori variano da 0 a 1.
  - **1**: Tutti i punti di una classe sono nel medesimo cluster.
  - **0**: Punti della stessa classe non sono stati raggruppati insieme.

### 5. **Fowlkes-Mallows Score**
- **Descrizione**: Misura la similarità tra i cluster previsti e le classi reali, è basato su precision e recall.
- **Valori**: I valori variano da 0 a 1.
  - **1**: Corrispondenza perfetta.
  - **0**: Nessuna corrispondenza.




In [ ]:
df_metrics = pd.DataFrame(df_metrics, columns=metriche_clustering)

# Calcolare la media lungo le colonne
media_colonne = df_metrics.mean(axis=0)

print(media_colonne)

### 2 Dimensions UMAP

In [23]:
import umap


embeddings_umap = umap.UMAP(densmap=True, n_neighbors=100, min_dist=0).fit_transform(embeddings)

In [ ]:
letters = df_medicatie['code5_ATC_code'].str[0].unique()

# lista di colori univoci
colors = plt.cm.rainbow(np.linspace(0, 1, len(letters)))

# crea un dizionario per mappare ogni lettera a un colore univoco
color_map = dict(zip(letters, colors))

plt.figure(figsize=[12, 12])

# plotta le prime due colonne del DataFrame
plt.scatter(embeddings_umap[:, 0], embeddings_umap[:, 1], \
    c=[color_map[lettera] for lettera in df_medicatie['code5_ATC_code'].str[0]])

# nomina i due assi
plt.xlabel('D1', fontsize=18)
plt.ylabel('D2', fontsize=18)

# crea la legenda
legend_elements = []
for lettera, colore in sorted(color_map.items()):
    legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', label=lettera, markerfacecolor=colore, markersize=10))

plt.legend(handles=legend_elements, loc='upper right', fontsize=10)


# Converti embeddings_pca in un DataFrame Pandas
df = pd.DataFrame(embeddings_umap, columns=['D1', 'D2'])

groups = df_medicatie['code5_ATC_code'].str[0]

df['group'] = groups
#df['color'] = df['group'].map(color_map)

df = df.sort_values(by='group')

fig = px.scatter(df, x='D1', y='D2', color='group')

# Imposto la grandezza dei marker
fig.update_traces(marker=dict(size=10))

# Rimuovi la griglia dagli assi
fig.update_layout(
    
    title=dict(
        text="2D Scatter plot",  # Il testo del titolo
        x=0.5,  # Centra il titolo
        xanchor='center',  # Imposta il punto di ancoraggio al centro
        font=dict(size=24)
    ),
    
    legend=dict(title=dict(text='ATC Group')),
    
    xaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    ),
    
    yaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    )
)

fig.show(renderer='browser')


## Standardized Dataset

In [7]:
embeddings = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\embeddings.parquet')

### General PCA

In [11]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(random_state=RS)
embeddings_pca = pca.fit_transform(embeddings)

exp_variance_ratio = pca.explained_variance_ratio_
exp_variance = pca.explained_variance_

exp_var_perc = [np.sum(exp_variance[:i+1]) for i in range(len(exp_variance))]/np.sum(exp_variance)

perc = 0
n_components = 0

while perc <= 0.9:
    perc += exp_variance_ratio[n_components]
    n_components += 1

embeddings_pca = pd.DataFrame(embeddings_pca[:, :n_components], columns=[f"PC{i+1}" for i in range(n_components)])

# Normalizzazione delle features
embeddings_pca = pd.DataFrame(normalize(embeddings_pca, norm='l2'), columns=embeddings_pca.columns)

embeddings_pca.insert(0, 'studyId_0831', df_medicatie['studyId_0831'])
embeddings_pca.insert(1, 'ATC_group', df_medicatie['code5_ATC_code'].str[0])

embeddings_pca.to_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\embeddings_perc_norm.parquet")

# Plot della figura
plt.figure(figsize = [20, 10])
plt.plot(np.arange(1, n_components+1), exp_var_perc[:n_components]*100, 'o-')

# crea una linea orizzontale in corrispondenza dell'ultimo punto
plt.axhline(y=perc*100, color='r', linestyle='--')

plt.xticks(np.concatenate((np.arange(1, n_components-2, 10), [n_components])))
plt.yticks(np.concatenate((np.arange(20, 100, 20), [round(perc*100, 1)])))
plt.tick_params(axis='both', labelsize=16)

plt.xlabel('Principal Components', fontsize=20)
plt.ylabel('Explained Variance [%]', fontsize=20)

plt.title(f'{n_components} components to capture {round(perc*100, 1):.1f}% of the informative content', fontsize=20)

plt.show()

## Clustering
Viene utilizzato il dataset standardizzato in seguito alla feature extraction con la PCA

In [1]:
import pandas as pd
import numpy as np


embeddings_perc = pd.read_parquet(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\parquet\embeddings_perc_norm.parquet')

In [2]:
from sklearn import metrics


metriche_clustering = [
    "adjusted_mutual_info_score",
    "adjusted_rand_score",
    "calinski_harabasz_score",
    "completeness_score",
#    "davies_bouldin_score",
    "fowlkes_mallows_score",
#    "homogeneity_score",
#    "silhouette_score",
]

# Numero di iterazioni del K-Means
n_iter = 100

### K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans


df_metrics = np.empty([n_iter, len(metriche_clustering)])

# Estraggo i veri cluster e li trasformo in numeri
true_labels = pd.factorize(embeddings_perc['ATC_group'])[0]

# Tolgo le prime due colonne dal dataframe originale
embeddings = embeddings_perc.iloc[:, 2:]

for i in range(n_iter):
    # Applicazione del K-Means con 14 cluster
    kmeans = KMeans(n_clusters=14)
    
    # Salvo i cluster predetti dal k-means
    pred_labels = kmeans.fit_predict(embeddings)
    
    df_metrics[i, :] = np.array([
        metrics.adjusted_mutual_info_score(true_labels, pred_labels),
        metrics.adjusted_rand_score(true_labels, pred_labels),
        metrics.calinski_harabasz_score(embeddings, pred_labels),
        metrics.completeness_score(true_labels, pred_labels),
        metrics.fowlkes_mallows_score(true_labels, pred_labels)
    ])   

df_metrics = pd.DataFrame(df_metrics, columns=metriche_clustering)

# Calcolare la media lungo le colonne
media_colonne = df_metrics.mean(axis=0)

media_colonne

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# Numero di colonne nel DataFrame
num_columns = len(df_metrics.columns)

# Calcola il numero di righe necessario per il subplot
num_rows = (num_columns + 2) // 3  # Aggiunge una riga se il numero di colonne non è un multiplo di 3

# Crea un subplot con il numero di righe e tre colonne
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))

# Itera attraverso ogni colonna
for i, column in enumerate(df_metrics.columns):
    ax = axes[i // 3, i % 3]  # Ottieni l'asse corretto
    
    # Traccia l'istogramma con un colore
    sns.histplot(df_metrics[column], bins=15, kde=False, stat="density", ax=ax, color='blue', alpha=0.3)
    
    # Traccia la linea di densità con un altro colore
    sns.kdeplot(df_metrics[column], ax=ax, color='DarkRed')
    
    ax.set_title(f'{column}')
    ax.set_ylabel('')
    ax.set_xlabel('')  # Rimuovi l'etichetta dell'asse x
    ax.grid(False)

# Rimuovi gli assi vuoti, se ce ne sono
if num_columns % 3 != 0:
    for j in range(num_columns, num_rows * 3):
        fig.delaxes(axes[j // 3, j % 3])

# Aggiungi un titolo all'intero subplot
plt.suptitle('Distribuzione delle metriche con K-Means clustering', fontsize=24)

plt.show()


### Random Selection

In [ ]:
# Estraggo i veri cluster e li trasformo in numeri
true_labels = pd.factorize(embeddings_perc['ATC_group'])[0]

# Tolgo le prime due colonne dal dataframe originale
embeddings = embeddings_perc.iloc[:, 2:]

df_metrics = np.empty([n_iter, len(metriche_clustering)])

for i in range(n_iter):
    random_labels = np.random.choice(true_labels, size=len(true_labels))
    
    df_metrics[i, :] = np.array([
        metrics.adjusted_mutual_info_score(true_labels, random_labels),
        metrics.adjusted_rand_score(true_labels, random_labels),
        metrics.calinski_harabasz_score(embeddings, random_labels),
        metrics.completeness_score(true_labels, random_labels),
        metrics.fowlkes_mallows_score(true_labels, random_labels)
    ])

df_metrics = pd.DataFrame(df_metrics, columns=metriche_clustering)

# Calcolare la media lungo le colonne
media_colonne = df_metrics.mean(axis=0)

media_colonne

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


# Numero di colonne nel DataFrame
num_columns = len(df_metrics.columns)

# Calcola il numero di righe necessario per il subplot
num_rows = (num_columns + 2) // 3  # Aggiunge una riga se il numero di colonne non è un multiplo di 3

# Crea un subplot con il numero di righe e tre colonne
fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))

# Itera attraverso ogni colonna
for i, column in enumerate(df_metrics.columns):
    ax = axes[i // 3, i % 3]  # Ottieni l'asse corretto
    
    # Traccia l'istogramma con un colore
    sns.histplot(df_metrics[column], bins=15, kde=False, stat="density", ax=ax, color='blue', alpha=0.3)
    
    # Traccia la linea di densità con un altro colore
    sns.kdeplot(df_metrics[column], ax=ax, color='DarkRed')
    
    ax.set_title(f'{column}')
    ax.set_ylabel('')
    ax.set_xlabel('')  # Rimuovi l'etichetta dell'asse x
    ax.grid(False)

# Rimuovi gli assi vuoti, se ce ne sono
if num_columns % 3 != 0:
    for j in range(num_columns, num_rows * 3):
        fig.delaxes(axes[j // 3, j % 3])
        

# Aggiungi un titolo all'intero subplot
plt.suptitle('Distribuzione delle metriche con selezione random', fontsize=24)

plt.show()

### DBScan

In [ ]:
from sklearn.cluster import DBSCAN


df_metrics = np.empty([n_iter, len(metriche_clustering)])

# Estraggo i veri cluster e li trasformo in numeri
true_labels = pd.factorize(embeddings_perc['ATC_group'])[0]

# Tolgo le prime due colonne dal dataframe originale
embeddings = embeddings_perc.iloc[:, 2:]

for i in range(n_iter):
    # Applicazione del K-Means con 14 cluster
    DBscan = DBSCAN(n_clusters=14)
    
    # Salvo i cluster predetti dal k-means
    pred_labels = DBscan.fit_predict(embeddings)
    
    df_metrics[i, :] = np.array([
        metrics.adjusted_mutual_info_score(true_labels, pred_labels),
        metrics.adjusted_rand_score(true_labels, pred_labels),
        metrics.calinski_harabasz_score(embeddings, pred_labels),
        metrics.completeness_score(true_labels, pred_labels),
        metrics.fowlkes_mallows_score(true_labels, pred_labels)
        ])   

df_metrics = pd.DataFrame(df_metrics, columns=metriche_clustering)

# Calcolare la media lungo le colonne
media_colonne = df_metrics.mean(axis=0)

print(media_colonne)

In [10]:
# Converti embeddings_pca in un DataFrame Pandas
df = pd.DataFrame(embeddings_pca, columns=['PC1', 'PC2'])

df['group'] = df_medicatie['code5_ATC_code'].str[0]

df = df.sort_values(by='group')

fig = px.scatter(df, x='PC1', y='PC2', color='group', color_discrete_map=color_map)

# Imposto la grandezza dei marker
fig.update_traces(marker=dict(size=7))

# Rimuovi la griglia dagli assi
fig.update_layout(
    
    title=dict(
        text="Scatter plot with the first two Principal Components",  # Il testo del titolo
        x=0.5,  # Centra il titolo
        xanchor='center',  # Imposta il punto di ancoraggio al centro
        font=dict(size=24)
    ),
    
    legend=dict(title=dict(text='ATC Group')),
    
    xaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    ),
    
    yaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    )
)

fig.show(renderer='browser')

fig.write_image(r"C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\twoPCA.png")

In [14]:
from sklearn.cluster import KMeans


# Applicazione del K-Means con 14 cluster
kmeans = KMeans(n_clusters=14, random_state=RS)
kmeans.fit(embeddings_pca)


# Aggiungere i cluster ottenuti al DataFrame originale
embeddings_pca['Cluster'] = kmeans.labels_

In [16]:
fig = px.scatter(embeddings_pca, x='PC1', y='PC2', color='Cluster', color_discrete_map=color_map)

# Imposto la grandezza dei marker
fig.update_traces(marker=dict(size=7))

# Rimuovi la griglia dagli assi
fig.update_layout(
    
    title=dict(
        text="Scatter plot with the first two Principal Components",  # Il testo del titolo
        x=0.5,  # Centra il titolo
        xanchor='center',  # Imposta il punto di ancoraggio al centro
        font=dict(size=24)
    ),
    
    legend=dict(title=dict(text='ATC Group')),
    
    xaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    ),
    
    yaxis=dict(
        showgrid=False, zeroline=False,
        title_font=dict(size=20)
    )
)

fig.show(renderer='browser')

fig.write_image(r"C:\Users\ddellamo\OneDrive - UMC Utrecht\Bureaublad\twoPCA.png")

Below insert medications with hierarchical code:
- How many A,B,C...?
- How many subgroup..?

Below insert medications with count

In [9]:
#df_medicatie_count = df_medicatie.drop_duplicates().sort_index().groupby('studyId_0831').agg(lambda x: len(x.to_list()))
#df_medicatie_count

In [10]:
#df_medicatie_count.reset_index().to_csv(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\csv\df_medicatie.csv', index=False)

In [9]:
C = "A01AB09"

In [ ]:
final_str = ''

In [ ]:
final_str += prova.query(f'atc_code == "{C[0]}"').atc_name + \
            prova.query(f'atc_code == "{C[0:3]}"').atc_name + \
            prova.query(f'atc_code == "{C[0:3]}"').atc_name
    